# Hunyuan3D-2mini — Textured Prop Generator (Aetheria Studio)

Generates **ONE textured 3D prop** from a **text prompt** for the Aetheria game (golem-fight clearing).
TripoSG is geometry-only — this adds the **texture** stage.

- **Model:** `tencent/Hunyuan3D-2mini` (0.6B DiT, subfolder `hunyuan3d-dit-v2-mini`) for shape + `tencent/Hunyuan3D-2` Paint pipeline for texture.
- **Text → image:** `HunyuanDiTPipeline` (`Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled`) — the repo's own text-to-3D route (same as `gradio_app.py --enable_t23d`).
- **VRAM (official README):** 6 GB shape, 16 GB shape+texture total — a free Colab **T4 (16 GB)** sits exactly on the limit, so the shape pipeline is freed (`del` + `empty_cache()`) before texture runs. If texture OOMs, the shape-only GLB is already saved.
- **Licence:** Tencent Hunyuan 3D 2.0 Community Licence — territory **excludes EU / UK / South Korea** (user is in Algeria, inside the Territory); the repo header also cites a non-commercial licence — treat outputs as **non-commercial / prototype only**, do not ship commercially without re-reading the full licence.
- **Runtime:** Colab GPU (T4). Run cells top to bottom. Output: `hunyuan3d-2mini-prop-textured.glb` (+ `hunyuan3d-2mini-prop-shape.glb` fallback).

In [ ]:
# Cell 1 — GPU check (needs a T4, 16 GB)
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), 'No GPU — switch Colab runtime to GPU (T4)'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB')
assert props.total_memory >= 14e9, 'Need >= 14 GB VRAM (T4 16 GB) for shape+texture'

In [ ]:
%%bash
# Cell 2 — Install from the official repo (verified against README + repo files)
set -e
if [ ! -d Hunyuan3D-2 ]; then git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git; fi
cd Hunyuan3D-2
pip install -q -r requirements.txt
pip install -q --force-reinstall "numpy==2.2.6"  # Kaggle fix: requirements.txt can leave a mixed/broken numpy on Python 3.13\npip install -q -e .
# Texture stage needs the compiled rasterizer + differentiable renderer (README)
cd hy3dgen/texgen/custom_rasterizer && python setup.py install && cd ../../..
cd hy3dgen/texgen/differentiable_renderer && python setup.py install && cd ../../..
echo INSTALL_OK

In [ ]:
# Cell 3 — Text prompt -> reference image (repo's own HunyuanDiT route)
import sys
sys.path.insert(0, 'Hunyuan3D-2')
from hy3dgen.text2image import HunyuanDiTPipeline

PROMPT = 'mossy stone lantern with carved runes, fantasy game prop'
t2i = HunyuanDiTPipeline('Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled', device='cuda')
image = t2i(PROMPT, seed=1234)  # PIL image, 1024x1024 (prompt truncated to 60 chars by the pipeline)
image.save('input-lantern.png')
image

In [ ]:
# Cell 4 — 2mini shape generation (0.6B, T4-safe) + shape-only fallback export
import torch
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini')
mesh = shape_pipe(image=image, num_inference_steps=30, octree_resolution=256)[0]
mesh.export('hunyuan3d-2mini-prop-shape.glb')
print(f'SHAPE: verts={len(mesh.vertices)} faces={len(mesh.faces)}')
# Free the shape model before texture: T4 has 16 GB and shape+texture needs 16 GB total
del shape_pipe
torch.cuda.empty_cache()

In [ ]:
# Cell 5 — Texture (Paint pipeline) + verify + download
import os
import trimesh
from hy3dgen.texgen import Hunyuan3DPaintPipeline

tex_pipe = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
mesh = tex_pipe(mesh, image=image)
mesh.export('hunyuan3d-2mini-prop-textured.glb')
size_mb = os.path.getsize('hunyuan3d-2mini-prop-textured.glb') / 1e6
print(f'TEXTURED GLB: {size_mb:.2f} MB')

# Verify: reload the GLB, check geometry + texture presence
check = trimesh.load('hunyuan3d-2mini-prop-textured.glb', force='scene')
geoms = list(check.geometry.values())
g = geoms[0]
has_uv = hasattr(g.visual, 'uv') and g.visual.uv is not None
print(f'VERIFY: verts={len(g.vertices)} faces={len(g.faces)} has_uv={has_uv}')
assert len(g.faces) > 0 and has_uv, 'Texture/UV missing — use the shape GLB + xAtlas instead'
print('HUNYUAN3D-2MINI TEST: PASS')

try:
    from google.colab import files
    files.download('hunyuan3d-2mini-prop-textured.glb')
except ImportError:
    print('Not on Colab — download the GLB from the notebook file browser instead')
